# 📊 財務分析ツール
Google Drive の PDF / Excel / Word ファイルを読み込んで財務指標を自動分析します。

**使い方:** 上から順に ▶️ ボタンを押して実行してください。

## ① 必要なライブラリをインストール
最初の1回だけ実行してください（2〜3分かかります）

In [ ]:
!pip install pymupdf openpyxl python-docx -q
print('✅ インストール完了')

## ② 財務分析コードを定義

In [ ]:
# ===== データモデル =====
from dataclasses import dataclass, field
from typing import Optional, Dict, List

@dataclass
class IncomeStatement:
    revenue: float
    cost_of_goods_sold: float
    gross_profit: Optional[float] = None
    selling_expenses: float = 0.0
    general_admin_expenses: float = 0.0
    operating_expenses: Optional[float] = None
    operating_income: Optional[float] = None
    non_operating_income: float = 0.0
    non_operating_expenses: float = 0.0
    ordinary_income: Optional[float] = None
    extraordinary_income: float = 0.0
    extraordinary_losses: float = 0.0
    income_before_tax: Optional[float] = None
    income_tax: float = 0.0
    net_income: Optional[float] = None
    period: str = ''

    def __post_init__(self):
        if self.gross_profit is None:
            self.gross_profit = self.revenue - self.cost_of_goods_sold
        if self.operating_expenses is None:
            self.operating_expenses = self.selling_expenses + self.general_admin_expenses
        if self.operating_income is None:
            self.operating_income = self.gross_profit - self.operating_expenses
        if self.ordinary_income is None:
            self.ordinary_income = self.operating_income + self.non_operating_income - self.non_operating_expenses
        if self.income_before_tax is None:
            self.income_before_tax = self.ordinary_income + self.extraordinary_income - self.extraordinary_losses
        if self.net_income is None:
            self.net_income = self.income_before_tax - self.income_tax

@dataclass
class BalanceSheet:
    cash_and_equivalents: float = 0.0
    accounts_receivable: float = 0.0
    inventory: float = 0.0
    other_current_assets: float = 0.0
    total_current_assets: Optional[float] = None
    property_plant_equipment: float = 0.0
    intangible_assets: float = 0.0
    investments: float = 0.0
    total_non_current_assets: Optional[float] = None
    total_assets: Optional[float] = None
    accounts_payable: float = 0.0
    short_term_debt: float = 0.0
    other_current_liabilities: float = 0.0
    total_current_liabilities: Optional[float] = None
    long_term_debt: float = 0.0
    other_non_current_liabilities: float = 0.0
    total_non_current_liabilities: Optional[float] = None
    total_liabilities: Optional[float] = None
    common_stock: float = 0.0
    retained_earnings: float = 0.0
    other_equity: float = 0.0
    total_equity: Optional[float] = None
    period: str = ''

    def __post_init__(self):
        if self.total_current_assets is None:
            self.total_current_assets = self.cash_and_equivalents + self.accounts_receivable + self.inventory + self.other_current_assets
        if self.total_non_current_assets is None:
            self.total_non_current_assets = self.property_plant_equipment + self.intangible_assets + self.investments
        if self.total_assets is None:
            self.total_assets = self.total_current_assets + self.total_non_current_assets
        if self.total_current_liabilities is None:
            self.total_current_liabilities = self.accounts_payable + self.short_term_debt + self.other_current_liabilities
        if self.total_non_current_liabilities is None:
            self.total_non_current_liabilities = self.long_term_debt + self.other_non_current_liabilities
        if self.total_liabilities is None:
            self.total_liabilities = self.total_current_liabilities + self.total_non_current_liabilities
        if self.total_equity is None:
            self.total_equity = self.common_stock + self.retained_earnings + self.other_equity

@dataclass
class FinancialData:
    company_name: str
    income_statement: IncomeStatement
    balance_sheet: BalanceSheet
    previous_income_statement: Optional[IncomeStatement] = None

print('✅ データモデル定義完了')

In [ ]:
# ===== ファイルローダー =====
import re
import os
import tempfile
from pathlib import Path

KEYWORD_MAP = {
    '売上高': 'revenue', '売上': 'revenue', '収益合計': 'revenue', '営業収益': 'revenue',
    '売上原価': 'cost_of_goods_sold', '原価合計': 'cost_of_goods_sold', '製造原価': 'cost_of_goods_sold',
    '売上総利益': 'gross_profit', '粗利': 'gross_profit', '粗利益': 'gross_profit',
    '販売費及び一般管理費': 'selling_and_admin', '販管費': 'selling_and_admin',
    '販売費': 'selling_expenses',
    '一般管理費': 'general_admin_expenses', '管理費': 'general_admin_expenses',
    '営業利益': 'operating_income', '営業損益': 'operating_income',
    '営業外収益': 'non_operating_income',
    '営業外費用': 'non_operating_expenses', '支払利息': 'non_operating_expenses',
    '経常利益': 'ordinary_income', '経常損益': 'ordinary_income',
    '特別利益': 'extraordinary_income',
    '特別損失': 'extraordinary_losses', '特別費用': 'extraordinary_losses',
    '税引前当期純利益': 'income_before_tax', '税引前利益': 'income_before_tax',
    '法人税': 'income_tax', '法人税等': 'income_tax',
    '当期純利益': 'net_income', '当期純損失': 'net_income', '純利益': 'net_income',
}

def _parse_number(text):
    if not text: return None
    t = str(text).strip().replace(',', '').replace('，', '').replace(' ', '')
    if not t or t in ('-', '―', '—', '－'): return 0.0
    neg = False
    if t.startswith(('△','▲','▽')): neg, t = True, t[1:]
    elif t.startswith('(') and t.endswith(')'): neg, t = True, t[1:-1]
    elif t.startswith('（') and t.endswith('）'): neg, t = True, t[1:-1]
    try:
        v = float(t)
        return -v if neg else v
    except: return None

def _match_keyword(label):
    label = label.strip()
    if label in KEYWORD_MAP: return KEYWORD_MAP[label]
    for kw in sorted(KEYWORD_MAP.keys(), key=len, reverse=True):
        if kw in label: return KEYWORD_MAP[kw]
    return None

def _build_income(fields, period, unit):
    if 'selling_and_admin' in fields and 'selling_expenses' not in fields:
        c = fields.pop('selling_and_admin')
        fields['selling_expenses'] = c * 0.5
        fields['general_admin_expenses'] = c * 0.5
    else:
        fields.pop('selling_and_admin', None)
    def g(k): return fields.get(k, 0.0)
    kw = dict(period=period, revenue=g('revenue'), cost_of_goods_sold=g('cost_of_goods_sold'),
              selling_expenses=g('selling_expenses'), general_admin_expenses=g('general_admin_expenses'),
              non_operating_income=g('non_operating_income'), non_operating_expenses=g('non_operating_expenses'),
              extraordinary_income=g('extraordinary_income'), extraordinary_losses=g('extraordinary_losses'),
              income_tax=g('income_tax'))
    for k in ('gross_profit','operating_income','ordinary_income','income_before_tax','net_income'):
        if k in fields: kw[k] = fields[k]
    return IncomeStatement(**kw)

def load_excel(path, period='', unit=1.0, sheet_name=None):
    import openpyxl
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    ws = wb[sheet_name] if sheet_name and sheet_name in wb.sheetnames else wb[wb.sheetnames[0]]
    for name in wb.sheetnames:
        if any(k in name for k in ('損益','PL','P&L','売上')):
            ws = wb[name]; break
    rows = [list(r) for r in ws.iter_rows(values_only=True)]
    wb.close()
    if not period:
        for row in rows[:5]:
            for c in row:
                m = re.search(r'(\d{4}年\d{1,2}月期)', str(c or ''))
                if m: period = m.group(1); break
    month_re = re.compile(r'\d{1,2}月')
    header_idx, header = 0, [str(c or '').strip() for c in rows[0]]
    for i, row in enumerate(rows[:20]):
        strs = [str(c or '').strip() for c in row]
        if sum(1 for s in strs if month_re.search(s)) >= 3 or any(k in ' '.join(strs) for k in ('合計','売上高','勘定科目')):
            header_idx, header = i, strs; break
    total_col = None
    for i, h in enumerate(header):
        if any(k in h for k in ('合計','累計','年計','通期','決算')): total_col = i; break
    if total_col is None:
        mc = [i for i, h in enumerate(header) if month_re.search(h)]
        if len(mc) >= 3: total_col = max(mc)
    fields = {}
    if total_col is not None:
        for row in rows[header_idx+1:]:
            label = str(row[0] or '').strip()
            f = _match_keyword(label)
            if f and total_col < len(row):
                v = _parse_number(row[total_col])
                if v is not None: fields.setdefault(f, v * unit)
    else:
        for row in rows[header_idx+1:]:
            label = str(row[0] or '').strip()
            f = _match_keyword(label)
            if f:
                for c in reversed(row[1:]):
                    v = _parse_number(c)
                    if v is not None: fields.setdefault(f, v * unit); break
    return _build_income(fields, period, unit)

def load_pdf(path, period='', unit=1.0):
    import fitz
    all_rows = []
    with fitz.open(path) as doc:
        for page in doc:
            words = page.get_text('words')
            pw = page.rect.width
            label_w, num_w = {}, {}
            for x0,y0,x1,y1,text,*_ in words:
                yk = int(y0/6)*6
                (label_w if x0 < pw*0.4 else num_w).setdefault(yk, []).append(text.strip())
            if not period:
                full = page.get_text()
                m = re.search(r'(\d{4}年\d{1,2}月期)', full)
                if m: period = m.group(1)
            for yk in sorted(label_w):
                label = ' '.join(label_w[yk])
                nums = num_w.get(yk, [])
                if not nums:
                    for dy in (6,12,-6,-12):
                        nums = num_w.get(yk+dy, [])
                        if nums: break
                total = nums[-1] if nums else ''
                all_rows.append([label, total])
    fields = {}
    for row in all_rows:
        f = _match_keyword(row[0])
        if f:
            v = _parse_number(row[1])
            if v is not None: fields.setdefault(f, v * unit)
    return _build_income(fields, period, unit)

def load_word(path, period='', unit=1.0):
    import docx
    doc = docx.Document(path)
    if not period:
        for p in doc.paragraphs[:10]:
            m = re.search(r'(\d{4}年\d{1,2}月期)', p.text)
            if m: period = m.group(1); break
    fields = {}
    for table in doc.tables:
        rows = [[c.text.strip() for c in row.cells] for row in table.rows]
        if not rows: continue
        header = rows[0]
        total_col = None
        month_re = re.compile(r'\d{1,2}月')
        for i, h in enumerate(header):
            if any(k in h for k in ('合計','累計','年計')): total_col = i; break
        if total_col is None:
            mc = [i for i, h in enumerate(header) if month_re.search(h)]
            if len(mc) >= 3: total_col = max(mc)
        for row in (rows[1:] if total_col is not None else rows):
            label = row[0]
            f = _match_keyword(label)
            if f:
                cells = row[1:] if total_col is None else [row[total_col]] if total_col < len(row) else []
                for c in reversed(cells):
                    v = _parse_number(c)
                    if v is not None: fields.setdefault(f, v * unit); break
    return _build_income(fields, period, unit)

def load_file(path, period='', unit=1.0, sheet_name=None):
    ext = Path(path).suffix.lower()
    if ext == '.pdf': return load_pdf(path, period, unit)
    if ext in ('.xlsx','.xls','.xlsm'): return load_excel(path, period, unit, sheet_name)
    if ext == '.docx': return load_word(path, period, unit)
    raise ValueError(f'非対応形式: {ext}')

print('✅ ファイルローダー定義完了')

In [ ]:
# ===== 分析・レポート出力 =====
def analyze_and_print(is_, company_name=''):
    sep = '=' * 55
    print(sep)
    print(f'  財務分析レポート: {company_name}')
    print(f'  対象期間: {is_.period}')
    print(sep)

    def pct(n, d): return (n/d*100) if d else 0.0

    print('\n【主要数値】')
    print(f'  売上高:       {is_.revenue:>15,.0f} 円')
    print(f'  売上原価:     {is_.cost_of_goods_sold:>15,.0f} 円')
    print(f'  売上総利益:   {is_.gross_profit:>15,.0f} 円')
    print(f'  営業利益:     {is_.operating_income:>15,.0f} 円')
    print(f'  経常利益:     {is_.ordinary_income:>15,.0f} 円')
    print(f'  当期純利益:   {is_.net_income:>15,.0f} 円')

    print('\n【収益性指標】')
    gm = pct(is_.gross_profit, is_.revenue)
    om = pct(is_.operating_income, is_.revenue)
    nm = pct(is_.net_income, is_.revenue)
    print(f'  売上総利益率: {gm:6.2f}%  ', '★★★' if gm>=50 else '★★' if gm>=30 else '★')
    print(f'  営業利益率:   {om:6.2f}%  ', '★★★' if om>=15 else '★★' if om>=5 else '★')
    print(f'  純利益率:     {nm:6.2f}%  ', '★★★' if nm>=10 else '★★' if nm>=3 else '★')

    print('\n【評価コメント】')
    if gm >= 50: print('  ・粗利率50%超 — 非常に高い価格競争力があります')
    elif gm >= 30: print('  ・粗利率は良好な水準です')
    else: print('  ・粗利率が低め。原価管理の見直しを検討してください')
    if om >= 15: print('  ・営業利益率15%超 — 優秀な収益力です')
    elif om >= 5: print('  ・営業利益率は安定した水準です')
    elif om >= 0: print('  ・営業利益率が低め。販管費の削減を検討してください')
    else: print('  ・営業損失が発生しています。早急な改善が必要です')
    print(sep)

print('✅ 分析関数定義完了')

## ③ Google Drive をマウント
「Googleドライブに接続」のポップアップが表示されたら **「Googleドライブに接続」** をタップしてください

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive マウント完了')

# マイドライブのファイル一覧を確認
import os
drive_root = '/content/drive/MyDrive'
print('\n📁 マイドライブのファイル一覧（上位20件）:')
for f in sorted(os.listdir(drive_root))[:20]:
    print(f'  {f}')

## ④ ファイルパスを設定して分析実行

上の一覧でファイル名を確認して、下の `FILE_PATH` を書き換えてください。

In [ ]:
# ==============================
# ここを書き換えてください
# ==============================
FILE_PATH    = '/content/drive/MyDrive/ファイル名.xlsx'  # ← ファイルのパス
COMPANY_NAME = '株式会社〇〇'                             # ← 企業名
PERIOD       = '2024年3月期'                              # ← 会計期間（空欄でも自動検出）
UNIT         = 1                                          # ← 単位: 1=円, 1000=千円, 1000000=百万円
SHEET_NAME   = None                                       # ← Excelシート名（None=自動）
# ==============================

print(f'読み込み中: {FILE_PATH}')
is_ = load_file(FILE_PATH, period=PERIOD, unit=UNIT, sheet_name=SHEET_NAME)
analyze_and_print(is_, company_name=COMPANY_NAME)

## ⑤ 複数ファイルをまとめて分析する場合

In [ ]:
# 複数ファイルをリストで指定
FILES = [
    {'path': '/content/drive/MyDrive/ファイル1.xlsx', 'company': '会社A', 'period': '2024年3月期', 'unit': 1},
    {'path': '/content/drive/MyDrive/ファイル2.pdf',  'company': '会社B', 'period': '2024年3月期', 'unit': 1000},
    {'path': '/content/drive/MyDrive/ファイル3.docx', 'company': '会社C', 'period': '2024年3月期', 'unit': 1},
]

for f in FILES:
    try:
        is_ = load_file(f['path'], period=f.get('period',''), unit=f.get('unit',1))
        analyze_and_print(is_, company_name=f['company'])
        print()
    except Exception as e:
        print(f"⚠️ {f['company']}: {e}")

## 🔍 うまく読み込めない場合のデバッグ

In [ ]:
# Excelの場合: シート名と内容を確認
import openpyxl
wb = openpyxl.load_workbook(FILE_PATH, read_only=True, data_only=True)
print('シート一覧:', wb.sheetnames)
ws = wb[wb.sheetnames[0]]
print('\n先頭10行:')
for i, row in enumerate(ws.iter_rows(values_only=True)):
    if i >= 10: break
    print(list(row))
wb.close()